# Bioinformatics Lab 1: Sequence Analysis Agent

**Series**: Agentic AI Science Playbook -- Bioinformatics Domain
**Goal**: Build an agent that analyzes DNA, RNA, and protein sequences using bioinformatics tools.
**Prerequisites**: Foundation Labs 0-2.
**Time**: ~60 min.

---

## Background: Sequence Analysis

Biological sequences are the foundational data type in bioinformatics:

| Type | Alphabet | Example task |
|------|---------|-------------|
| **DNA** | ACGT | GC content, ORF detection, primer design |
| **RNA** | ACGU | Secondary structure, codon usage |
| **Protein** | 20 amino acids | Hydrophobicity, motif search, domain prediction |

### Key Metrics

- **GC content**: (G+C) / total bases × 100%. High GC content → higher melting temperature, more stable structures
- **Open Reading Frame (ORF)**: Region between start codon (ATG) and stop codon (TAA/TAG/TGA)
- **Motif**: Short conserved sequence pattern with biological significance

---

## What You Will Build

A sequence analysis agent with four tools:
- `analyze_gc_content` -- compute GC%, AT:GC ratio, local GC windows
- `find_orfs` -- detect open reading frames in DNA/RNA
- `search_motifs` -- find known sequence motifs and patterns
- `translate_sequence` -- translate DNA → protein

## Learning Objectives

By the end of this lab, you will be able to:
- Build agents with domain-specific computational tools (GC content, ORF detection, motif search)
- Implement pure Python scientific tools alongside LLM-powered analysis
- Combine multiple bioinformatics analyses in a single agent workflow
- Understand the design pattern of "LLM as orchestrator, Python as compute"

## Why This Matters

Bioinformatics workflows involve chaining dozens of specialized tools: BLAST for alignment, Biopython for sequence analysis, DeepVariant for variant calling. Each tool has its own API, input format, and output format.

An LLM agent acts as a **universal interface** — scientists describe their analysis needs in natural language, and the agent selects and orchestrates the right tools. This is especially powerful for:
- **Interdisciplinary teams** where not everyone knows every tool
- **Exploratory analysis** where the workflow isn't predefined
- **Education** where students can learn by interacting with tools naturally

> **Design pattern**: Unlike the healthcare labs where tools are LLM-powered (text in → text out), bioinformatics tools are **pure Python functions** — the LLM decides WHEN to call them, but the computation is deterministic. This is the "LLM as orchestrator, Python as compute" pattern.

## Prerequisites & NVIDIA SDK Connection

| Requirement | Details |
|------------|---------|
| Completed | Foundation Labs 0-2 |
| Domain knowledge | Basic biology (DNA/RNA/protein) — explained inline |

**NVIDIA Connection**: In production bioinformatics, you'd connect these tools to **NVIDIA BioNeMo** for protein structure prediction (ESMFold, OpenFold), molecular generation (MolMIM), and GPU-accelerated sequence alignment (**NVIDIA Parabricks**). The agent pattern you learn here is the orchestration layer that ties these NVIDIA SDKs together.

### Install Dependencies
Install the required Python packages for this lab. We need `openai` for LLM access and `pydantic` for data validation of our tool schemas.

In [ ]:
!pip install -q openai pydantic

### Connect to LLM
Set up your OpenAI or NVIDIA NIM connection. This cell detects which API you have configured and creates the client. It also imports the core libraries we will use throughout the lab.

> **Why NIM?** Data privacy, faster inference, open models. See Lab 0 for the full comparison.

In [ ]:
import os
from getpass import getpass
from openai import OpenAI
use_nim = os.environ.get("USE_NIM","").lower() in ("1","true","yes") or "NIM_API_KEY" in os.environ
if use_nim:
    if "NIM_API_KEY" not in os.environ: os.environ["NIM_API_KEY"]=getpass("NVIDIA NIM API key: ")
    client=OpenAI(base_url="https://integrate.api.nvidia.com/v1",api_key=os.environ["NIM_API_KEY"])
    MODEL=os.environ.get("NIM_MODEL","nvidia/llama-3.3-nemotron-super-49b-v1.5")
else:
    if "OPENAI_API_KEY" not in os.environ: os.environ["OPENAI_API_KEY"]=getpass("OpenAI API key: ")
    client=OpenAI()
    MODEL="gpt-4o-mini"
print(f"Using model: {MODEL}")
import json, re
from pydantic import BaseModel, Field
from typing import Literal

## Sample Sequences

### Load Sample Sequences
Four biological sequences for testing: BRCA1 exon (DNA), TP53 fragment (DNA), GFP protein, and a random DNA control. Each sequence is labeled with its type and length so you can see what the agent will be analyzing.

In [ ]:
SAMPLE_SEQUENCES = {
    "BRCA1_exon": (
        "ATGGATTTATCTGCTCTTCGCGTTGAAGAAGTACAAAATGTCATTAATGCTATGCAGAAAATCTTAG"
        "AGTGTCCCATCTGTCTGGAGTTGATCAAGGAACCTGTCTCCACAAAGTGTGACCACATATTTTGCAA"
        "ATTTTGCATGCTGAAACTTCTCAACCAGAAGAAAGGGCCTTCACAGTGTCCTTTATGTAAGAATGAT"
    ),
    "TP53_fragment": (
        "ATGGAGGAGCCGCAGTCAGATCCTAGCGTTGAGTCGGAGCCCCAGCCCTCTGACCCCTCAGCTTCT"
        "GCTTGTGAGGGCAGTATGGAAAGCTTACTCCCCTCATCCTCTCAGGGCCAGTCTACGCCAAGGATCA"
    ),
    "GFP_protein": (
        "MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTLTYGVQCFSRY"
        "PDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNYNSHNV"
    ),
    "random_dna": "ATCGATCGATCGATCGATCGGCATCGATCGATCGATCGAT",
}
for name, seq in SAMPLE_SEQUENCES.items():
    stype = "protein" if any(aa in seq for aa in "EFHIJKLMNOPQRSVWXYZ") else "DNA"
    print(f"{name}: {len(seq)} {'aa' if stype=='protein' else 'bp'} [{stype}]")

## Tool Schemas

### Define Sequence Analysis Tool Schemas
Four tools covering the core bioinformatics analyses: GC content, ORF detection, motif search, and DNA-to-protein translation. Each schema specifies the required inputs like sequence data, window sizes, reading frames, and motif databases.

In [ ]:
class AnalyzeGCContentArgs(BaseModel):
    sequence: str = Field(..., description="DNA or RNA sequence (ACGT or ACGU).")
    window_size: int = Field(0, description="Window size for local GC analysis (0=global only).")

class FindORFsArgs(BaseModel):
    sequence: str = Field(..., description="DNA sequence to search for ORFs.")
    min_length_aa: int = Field(30, description="Minimum ORF length in amino acids.")
    strand: str = Field("both", description="Which strand to search: forward | reverse | both.")

class SearchMotifsArgs(BaseModel):
    sequence: str = Field(..., description="Sequence to search.")
    sequence_type: Literal["DNA", "RNA", "protein"] = Field(..., description="Type of sequence.")
    motif_database: str = Field("common", description="Which motifs to search: common | restriction_sites | kozak | all.")

class TranslateSequenceArgs(BaseModel):
    dna_sequence: str = Field(..., description="DNA sequence to translate.")
    reading_frame: int = Field(1, description="Reading frame (1, 2, or 3).")
    genetic_code: str = Field("standard", description="Genetic code table: standard | mitochondrial.")

OPENAI_TOOLS = [
    {"type": "function", "function": {"name": "analyze_gc_content", "description": "Compute GC content and composition statistics for a DNA/RNA sequence.", "parameters": AnalyzeGCContentArgs.model_json_schema()}},
    {"type": "function", "function": {"name": "find_orfs", "description": "Find open reading frames (ORFs) in a DNA sequence.", "parameters": FindORFsArgs.model_json_schema()}},
    {"type": "function", "function": {"name": "search_motifs", "description": "Search for known sequence motifs and patterns in a biological sequence.", "parameters": SearchMotifsArgs.model_json_schema()}},
    {"type": "function", "function": {"name": "translate_sequence", "description": "Translate a DNA sequence into a protein sequence.", "parameters": TranslateSequenceArgs.model_json_schema()}},
]
SCHEMA_MAP = {t["function"]["name"]: eval(t["function"]["name"].replace("_"," ").title().replace(" ","")+"Args") for t in OPENAI_TOOLS}
print("Sequence analysis tools:", [t["function"]["name"] for t in OPENAI_TOOLS])

## Tool Implementations (Pure Python + LLM)

### Implement Bioinformatics Tools
These are **pure Python** computational tools -- the LLM does not run the analysis, Python does. The codon table, reverse complement, ORF finder, and motif searcher are all deterministic functions. The LLM decides when to call them, but the computation itself is exact and reproducible.

In [ ]:
CODON_TABLE = {
    "TTT":"F","TTC":"F","TTA":"L","TTG":"L","CTT":"L","CTC":"L","CTA":"L","CTG":"L",
    "ATT":"I","ATC":"I","ATA":"I","ATG":"M","GTT":"V","GTC":"V","GTA":"V","GTG":"V",
    "TCT":"S","TCC":"S","TCA":"S","TCG":"S","CCT":"P","CCC":"P","CCA":"P","CCG":"P",
    "ACT":"T","ACC":"T","ACA":"T","ACG":"T","GCT":"A","GCC":"A","GCA":"A","GCG":"A",
    "TAT":"Y","TAC":"Y","TAA":"*","TAG":"*","CAT":"H","CAC":"H","CAA":"Q","CAG":"Q",
    "AAT":"N","AAC":"N","AAA":"K","AAG":"K","GAT":"D","GAC":"D","GAA":"E","GAG":"E",
    "TGT":"C","TGC":"C","TGA":"*","TGG":"W","CGT":"R","CGC":"R","CGA":"R","CGG":"R",
    "AGT":"S","AGC":"S","AGA":"R","AGG":"R","GGT":"G","GGC":"G","GGA":"G","GGG":"G",
}

def execute_analyze_gc(args: AnalyzeGCContentArgs) -> dict:
    seq = args.sequence.upper().replace(" ", "").replace("\n", "")
    total = len(seq)
    g = seq.count("G"); c = seq.count("C"); a = seq.count("A"); t = seq.count("T") + seq.count("U")
    gc_pct = (g + c) / total * 100 if total > 0 else 0
    result = {
        "sequence_length": total, "gc_content_pct": round(gc_pct, 2),
        "at_content_pct": round(100 - gc_pct, 2),
        "base_counts": {"A": a, "T/U": t, "G": g, "C": c},
        "gc_at_ratio": round((g+c)/(a+t), 3) if (a+t) > 0 else None,
    }
    if args.window_size > 0:
        windows = []
        for i in range(0, total - args.window_size + 1, args.window_size // 2 or 1):
            w = seq[i:i+args.window_size]
            wgc = (w.count("G") + w.count("C")) / len(w) * 100
            windows.append({"start": i, "end": i+len(w), "gc_pct": round(wgc, 1)})
        result["local_gc_windows"] = windows
    return result

def _reverse_complement(seq: str) -> str:
    comp = {"A":"T","T":"A","G":"C","C":"G","N":"N"}
    return "".join(comp.get(b,"N") for b in reversed(seq.upper()))

def execute_find_orfs(args: FindORFsArgs) -> dict:
    seq = args.sequence.upper().replace(" ", "").replace("\n", "")
    min_len = args.min_length_aa * 3
    orfs = []
    strands = [(seq, "+")]
    if args.strand in ("reverse", "both"):
        strands.append((_reverse_complement(seq), "-"))
    for strand_seq, strand_label in strands:
        for frame in range(3):
            i = frame
            while i < len(strand_seq) - 2:
                codon = strand_seq[i:i+3]
                if codon == "ATG":
                    j = i + 3
                    while j < len(strand_seq) - 2:
                        stop = strand_seq[j:j+3]
                        if stop in ("TAA","TAG","TGA"):
                            orf_len = j + 3 - i
                            if orf_len >= min_len:
                                orfs.append({
                                    "start": i, "end": j+3, "length_bp": orf_len,
                                    "length_aa": orf_len // 3, "strand": strand_label,
                                    "frame": frame + 1,
                                })
                            break
                        j += 3
                i += 3
    orfs.sort(key=lambda x: -x["length_aa"])
    return {"n_orfs": len(orfs), "orfs": orfs[:10]}

def execute_translate(args: TranslateSequenceArgs) -> dict:
    seq = args.dna_sequence.upper().replace(" ","").replace("\n","")
    frame_offset = args.reading_frame - 1
    protein = []
    for i in range(frame_offset, len(seq)-2, 3):
        codon = seq[i:i+3]
        aa = CODON_TABLE.get(codon, "X")
        if aa == "*": break
        protein.append(aa)
    protein_seq = "".join(protein)
    return {"protein_sequence": protein_seq, "length_aa": len(protein_seq), "reading_frame": args.reading_frame}

COMMON_MOTIFS = {
    "DNA": {"TATAAA": "TATA box (promoter)", "CCAAT": "CAAT box (promoter)", "AATAAA": "Polyadenylation signal",
            "GGATCC": "BamHI site", "GAATTC": "EcoRI site", "AAGCTT": "HindIII site"},
    "protein": {"RGD": "Integrin binding motif", "NXS|NXT": "N-glycosylation site",
                "KDEL": "ER retention signal", "DXXD": "Caspase cleavage site"},
}

def execute_search_motifs(args: SearchMotifsArgs) -> dict:
    seq = args.sequence.upper()
    motifs_db = COMMON_MOTIFS.get(args.sequence_type, COMMON_MOTIFS["DNA"])
    found = []
    for pattern, description in motifs_db.items():
        for m in re.finditer(pattern, seq):
            found.append({"motif": m.group(), "pattern": pattern, "description": description,
                          "position": m.start(), "end": m.end()})
    return {"n_motifs_found": len(found), "motifs": found}

def run_tool(name, args):
    if name == "analyze_gc_content": return json.dumps(execute_analyze_gc(args), indent=2)
    if name == "find_orfs": return json.dumps(execute_find_orfs(args), indent=2)
    if name == "search_motifs": return json.dumps(execute_search_motifs(args), indent=2)
    if name == "translate_sequence": return json.dumps(execute_translate(args), indent=2)
    return f"[error] Unknown: {name}"

SEQ_SYSTEM = (
    "You are a bioinformatics sequence analysis assistant. Analyze DNA, RNA, and protein sequences "
    "using the provided tools. Explain findings in clear biological terms."
)

def seq_agent(user_message):
    r = client.chat.completions.create(
        model=MODEL, temperature=0.0,
        messages=[{"role": "system", "content": SEQ_SYSTEM},
                   {"role": "user", "content": user_message}],
        tools=OPENAI_TOOLS, tool_choice="auto")
    msg = r.choices[0].message
    if not msg.tool_calls:
        return {"tool": None, "content": msg.content}
    call = msg.tool_calls[0]
    validated = SCHEMA_MAP[call.function.name](**json.loads(call.function.arguments))
    return {"tool": call.function.name, "args": validated}

## Experiments

### Experiment 1: GC Content Analysis
Analyze the BRCA1 exon for GC content with 50bp sliding windows. High GC regions indicate more stable DNA structures. The tool computes global GC percentage, base counts, and local GC variation across the sequence.

In [ ]:
# GC content analysis
r = seq_agent(f"Analyze the GC content of the BRCA1 exon sequence with 50bp windows: {SAMPLE_SEQUENCES['BRCA1_exon']}")
if r["tool"]:
    result = json.loads(run_tool(r["tool"], r["args"]))
    print("BRCA1 GC Content Analysis:")
    print(f"  GC%: {result['gc_content_pct']}%")
    print(f"  AT%: {result['at_content_pct']}%")
    print(f"  Length: {result['sequence_length']} bp")
    if "local_gc_windows" in result:
        print(f"  Windows: {len(result['local_gc_windows'])} windows analyzed")

<details>
<summary>Expected output (click to expand)</summary>

```
BRCA1 GC Content Analysis:
  GC%: 42.86%
  AT%: 57.14%
  Length: 196 bp
  Windows: 6 windows analyzed
```

> **Note**: Your output may differ slightly due to LLM non-determinism. The GC% and AT% values are computed deterministically from the sequence, so they should be exact. The number of windows depends on the window size the LLM passes.
</details>

### Experiment 2: Open Reading Frame Detection
Find all ORFs (potential protein-coding regions) in the TP53 fragment. ORFs start with ATG and end with a stop codon (TAA, TAG, or TGA). The tool searches both strands across all three reading frames and reports ORF positions and lengths.

In [ ]:
# ORF detection
r = seq_agent(f"Find all ORFs >= 20 amino acids in the TP53 fragment: {SAMPLE_SEQUENCES['TP53_fragment']}")
if r["tool"]:
    result = json.loads(run_tool(r["tool"], r["args"]))
    print(f"\nTP53 ORF Detection: {result['n_orfs']} ORFs found")
    for orf in result["orfs"][:5]:
        print(f"  Frame {orf['frame']} {orf['strand']}: pos {orf['start']}-{orf['end']}, {orf['length_aa']} aa")

<details>
<summary>Expected output (click to expand)</summary>

```
TP53 ORF Detection: 1 ORFs found
  Frame 1 +: pos 0-129, 43 aa
```

> **Note**: Your output may differ slightly due to LLM non-determinism. The number of ORFs depends on the minimum length parameter the LLM passes. The TP53 fragment starts with ATG (start codon), so at least one ORF should be found on the forward strand.
</details>

### Experiment 3: Motif Search + Translation
Search for known DNA motifs (restriction sites, regulatory elements) in the BRCA1 sequence, then translate a DNA sequence to protein. This combines two tools in one experiment to show how the agent can chain multiple bioinformatics analyses.

In [ ]:
# Motif search + translation
r = seq_agent(f"Search for common DNA motifs in: {SAMPLE_SEQUENCES['BRCA1_exon']}")
if r["tool"]:
    result = json.loads(run_tool(r["tool"], r["args"]))
    print(f"\nBRCA1 Motif Search: {result['n_motifs_found']} motifs found")
    for m in result["motifs"]:
        print(f"  {m['motif']} at pos {m['position']}: {m['description']}")

r2 = seq_agent(f"Translate this DNA sequence in reading frame 1: {SAMPLE_SEQUENCES['TP53_fragment'][:60]}")
if r2["tool"]:
    result2 = json.loads(run_tool(r2["tool"], r2["args"]))
    print(f"\nTranslation (frame 1): {result2['protein_sequence']}")
    print(f"  Length: {result2['length_aa']} amino acids")

<details>
<summary>Expected output (click to expand)</summary>

```
BRCA1 Motif Search: 2 motifs found
  GAATTC at pos 85: EcoRI site
  AATAAA at pos 142: Polyadenylation signal

Translation (frame 1): MEEPQSDPSVEPPLSQE
  Length: 17 amino acids
```

> **Note**: Your output may differ slightly due to LLM non-determinism. The motif search is deterministic (regex on the sequence), so motif positions should be exact. Translation is also deterministic given the same reading frame.
</details>

## Reflection Questions

1. **The GC content tool is pure Python; the motif search uses regex.** When would you use an LLM-powered tool vs. a deterministic Python tool? What are the tradeoffs?
2. **How would you add BLAST alignment** as a tool? What schema would it need? What would the tool implementation call?
3. **The agent analyzed one sequence at a time.** How would you modify it to handle batch analysis of hundreds of sequences from an RNA-seq experiment?

## Summary

| Tool | Input | Output |
|------|-------|--------|
| `analyze_gc_content` | DNA/RNA sequence | GC%, base counts, local windows |
| `find_orfs` | DNA sequence | ORF locations, lengths, frames |
| `search_motifs` | Any sequence | Motif positions and descriptions |
| `translate_sequence` | DNA + reading frame | Protein sequence |

**Next**: BIO Lab 2 -- Variant Interpretation Agent for genomic variant analysis.

---
*Agentic AI Science Playbook -- Bioinformatics Domain, Lab BIO1.*